# Clase 2 — Componer funciones: dibujar lo imposible

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Con un montón de piezas tontas y un doblez
> entre pieza y pieza se dibuja cualquier figura. Eso, y no otra cosa, es una
> red neuronal.

---

**Cuando termines este notebook vas a poder:**

1. Componer funciones en Python y explicar por qué el **orden importa**.
2. Escribir una rampa (ReLU) en una línea y usarla para armar una V invertida y una meseta.
3. Escribir **una red neuronal entera, a mano, en seis líneas**, sin PyTorch y sin nada.
4. Leer la versión matricial de esa misma red y comprobar que da **exactamente** lo mismo.

**Lo que hace falta traer:** la clase 1 (variable vs parámetro) y las ganas de
mover números hasta que salga.

**Cómo se usa este notebook.** Corré las celdas en orden, de arriba hacia abajo
(`Shift + Enter`). Donde diga `# TODO` te toca a vos: el notebook viene con la
respuesta **puesta mal a propósito**, así que las celdas de verificación te van
a decir "todavía no" hasta que la arregles. Eso es lo esperado, no es que se
rompió nada.

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el
repo y deja todo a mano; en tu máquina no hace prácticamente nada porque ya
tenés los archivos. Corrémela y seguí de largo: no hay nada que entender acá.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys

REPO = "https://github.com/johannperez/dlss"
RAMA = "claude/dl-course-social-scientists-279dd2"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                        REPO, CARPETA], check=True)
    raiz = buscar_raiz(CARPETA)

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab: revisá que la celda haya podido clonar (necesita internet).\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`."
    )

# Dependencias: en Colab casi todas vienen puestas. Instalamos sólo lo que falte.
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],
                       check=True)

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports, semilla, y un ayudante

`ssdl` es el paquete del curso: la plomería que no queremos repetir en diez
notebooks. No esconde nada importante — está en `_shared/ssdl/` y se puede leer.

Fijamos la semilla en 1983 **siempre**. No hace al modelo mejor: hace a tu
resultado reproducible, que es condición para poder afirmar algo.

La función `revisar` de abajo es nuestro corrector: **no rompe el notebook**,
te dice qué mirar.

In [ ]:
import numpy as np

from ssdl import datos, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso


def revisar(condicion, si_va, si_no):
    """Un chequeo amable. Si no te sale, te dice qué mirar; nunca explota."""
    if condicion:
        print("✓ " + si_va)
    else:
        print("✗ todavía no —", si_no)


print("Listo. A trabajar.")

---

## 1. El desafío: la figura que una recta no puede dibujar

Estos son datos **reales**: qué porcentaje de las palabras del discurso
presidencial de apertura de sesiones pertenecen al léxico de *inflación*, año
por año, de 1984 a 2019. Es una media móvil de 3 años, así que lo que estás
mirando es "en qué parte del período el tema estuvo en agenda".

> **Antes de correr, apostá.** ¿Qué forma tiene? ¿Sube, baja, va y viene?
> Escribilo mentalmente. Después mirá.

In [ ]:
# La serie de la clase 2: dos picos, un valle, y un pico grande al final.
# `suavizada=True` es la media móvil de 3 años (columna `inflacion_suave`).
anios, inflacion = datos.serie_menciones("inflacion", desde=1984, hasta=2019,
                                         suavizada=True)

print(f"{len(anios)} años, de {int(anios[0])} a {int(anios[-1])}")
print(f"El máximo fue en {int(anios[inflacion.argmax()])}: "
      f"{inflacion.max():.4f}% de las palabras del discurso.")
print(f"El mínimo fue en {int(anios[inflacion.argmin()])}: "
      f"{inflacion.min():.4f}%.")

g.plot_serie(anios, inflacion, etiqueta="menciones a inflación",
             titulo="El desafío de hoy: dibujá esto con una función")

### El fracaso reproducible

Ahora la mejor herramienta de la clase 1 —la recta con sus dos perillas— contra
esta curva. Y no una recta cualquiera: **la mejor recta posible**. Para las
rectas existe una fórmula cerrada que da el mínimo exacto, y `numpy` la trae:
`np.polyfit`. No hay que buscar nada a ojo.

In [ ]:
# np.polyfit con grado 1 devuelve la pendiente y la ordenada de la MEJOR recta
# posible: la que hace más chico el promedio de los errores al cuadrado.
pendiente, ordenada = np.polyfit(anios, inflacion, 1)

def mejor_recta(x):
    return pendiente * x + ordenada

mse_recta = float(np.mean((inflacion - mejor_recta(anios)) ** 2))
mse_promedio = float(np.mean((inflacion - inflacion.mean()) ** 2))

print(f"La mejor recta sube {pendiente:.5f} puntos por año.")
print(f"Su desajuste (MSE) es      {mse_recta:.5f}")
print(f"El de una raya horizontal  {mse_promedio:.5f}   <- ¡casi lo mismo!")
print(f"\nLa mejor recta posible explica el "
      f"{100 * (1 - mse_recta / mse_promedio):.1f} % de la variación.")

g.plot_ajuste(anios, inflacion, modelo=mejor_recta, loss=mse_recta,
              xlabel="año", ylabel="% de palabras",
              etiqueta_modelo="la mejor recta posible",
              titulo="No es que la ajustemos mal. Es que no se puede.")

Leelo despacio, porque es el gancho de la clase: **la mejor recta posible
explica el 1,3 % de la variación**. Dibujar una raya horizontal en el promedio,
sin mirar el año, da prácticamente lo mismo.

### "¿Y si le doy más perillas?"

Es la reacción natural, y está mal. Probemos: en vez de `a·x + b`, pongamos
`a·x + b + c`, y después `a·x + b + c·1 + d·2 + e·x`. Cinco perillas.

In [ ]:
# Cinco perillas, puestas en cualquier lado
a, b, c, d, e = 0.0004, 0.05, 0.007, 0.002, 0.00003

def recta_con_cinco_perillas(x):
    return a * x + b + c * 1 + d * 2 + e * x

# Ahora colapsamos las cinco a mano: agrupamos lo que multiplica a x y lo que no
pendiente_total = a + e
ordenada_total = b + c * 1 + d * 2

def la_misma_recta_con_dos(x):
    return pendiente_total * x + ordenada_total

iguales = np.allclose(recta_con_cinco_perillas(anios), la_misma_recta_con_dos(anios))

print(f"con 5 perillas: y = {a}·x + {b} + {c}·1 + {d}·2 + {e}·x")
print(f"con 2 perillas: y = {pendiente_total:.5f}·x + {ordenada_total:.3f}")
print(f"\n¿dibujan exactamente la misma figura?  {iguales}")
print("\nMás parámetros NO es más flexibilidad. Acordate de esto cuando alguien")
print("te venda un modelo por su tamaño.")

---

## 2. Componer: la salida de una entra a la otra

Si una máquina no alcanza, ponemos dos. Nada más que eso.

**Vos ya componés funciones**, todos los días y en papel: respuestas crudas →
recodificación de categorías → índice sumado → estandarización → tipología.
Cada flecha es una función. Y como en una red: si una etapa distorsiona, todo
lo que viene después hereda la distorsión.

In [ ]:
def doble(x):
    return 2 * x

def mas_diez(x):
    return x + 10

# Se lee de ADENTRO hacia AFUERA, como los paréntesis de siempre.
print("doble(mas_diez(3))  =", doble(mas_diez(3)), "  ->  primero suma, después duplica")
print("mas_diez(doble(3))  =", mas_diez(doble(3)), "  ->  primero duplica, después suma")
print()
print("Mismas dos funciones. Mismos números. Otro resultado.")
print("Si componer fuera sumar, el orden no importaría. Componer no es sumar.")

### `TODO 1` — Tu tercera función

Escribí una tercera función y componé las tres.

**Antes de correr la celda, predecí el resultado y anotalo en el comentario que
dice `# mi predicción:`.** No es una formalidad: la predicción fallida es donde
se aprende. Sin escribirla primero, correr la celda es mirar televisión.

*Pista: resolvé de adentro hacia afuera, como paréntesis. El resultado de la de
más adentro es lo que le entra a la que está afuera.*

In [ ]:
def menos_cinco(x):
    return x - 5      # TODO: cambiala por la operación que quieras


# mi predicción: doble(mas_diez(menos_cinco(4))) va a dar ....... (escribila acá)
# mi predicción: menos_cinco(mas_diez(doble(4))) va a dar ....... (y acá)

resultado_1 = doble(mas_diez(menos_cinco(4)))
resultado_2 = menos_cinco(mas_diez(doble(4)))

print("doble(mas_diez(menos_cinco(4))) =", resultado_1)
print("menos_cinco(mas_diez(doble(4))) =", resultado_2)

revisar(resultado_1 != resultado_2,
        "dan distinto, que es todo el punto: el orden cambia el resultado.",
        "te dieron iguales. Puede pasar con algunas combinaciones "
        "(probá con otra operación): que a veces coincidan no quiere decir "
        "que el orden no importe.")

---

## 3. La rampa: la función más tonta del mundo

Encadenar rectas no sirve de nada: dos estiramientos seguidos siguen siendo un
estiramiento. Hace falta algo que **doble**. Y va a ser ridículamente simple:

> **si es negativo, poné cero. Si es positivo, dejalo pasar.**

Eso es `ReLU` (*rectified linear unit*, "unidad lineal rectificada", un nombre
espantoso para `max(0, x)`). Le agregamos dos perillas: **dónde está el codo**
y **qué tan empinada** es.

Es una línea de código. Subrayado: **una línea**.

In [ ]:
def rampa(x, codo=0.0, altura=1.0):
    """Plana hasta el codo; después sube (o baja, si la altura es negativa)."""
    return altura * np.maximum(0, x - codo)


x = np.linspace(0, 10, 400)

fig, ax = g.figura(9, 4.2, titulo="Tres rampas: mismo mecanismo, distintas perillas",
                   xlabel="x", ylabel="rampa(x)")
g.plot_funcion(lambda t: rampa(t, codo=2, altura=1.0), x=x, ax=ax,
               etiqueta="codo en 2, altura 1", color=g.PARAMETRO)
g.plot_funcion(lambda t: rampa(t, codo=5, altura=2.0), x=x, ax=ax,
               etiqueta="codo en 5, altura 2", color=g.ACENTO)
g.plot_funcion(lambda t: rampa(t, codo=7, altura=-1.5), x=x, ax=ax,
               etiqueta="codo en 7, altura −1,5", color=g.MODELO)
ax.legend()
print("Fijate: antes del codo, cero. Después del codo, una recta. Y nada más.")

### `TODO 2` — Una V invertida con dos rampas

Sumá **dos** rampas para que la curva **suba y después baje**, formando una V
invertida (un pico). Movés `codo_2` y `altura_2` hasta que salga.

*Pista: la segunda rampa tiene que tener altura **negativa**. Y si querés que
después del pico la curva baje de verdad, esa altura negativa tiene que ser en
valor absoluto **más grande** que la de la primera. Probá con el doble.*

Se resuelve por prueba y error, y está perfecto que así sea.

In [ ]:
codo_1, altura_1 = 2.0, 1.0
codo_2, altura_2 = 5.0, 0.5      # TODO: tocá estos dos números


def v_invertida(x):
    return rampa(x, codo_1, altura_1) + rampa(x, codo_2, altura_2)


y = v_invertida(x)
g.plot_funcion(v_invertida, x=x, xlabel="x", ylabel="suma de las dos rampas",
               titulo="TODO 2: ¿ya sube y después baja?")

sube_antes = y[x < codo_2].max() > y[0] + 1e-9
baja_despues = y[-1] < y[x < codo_2].max() - 1e-9
revisar(sube_antes and baja_despues,
        f"listo: sube hasta {y.max():.2f} y después baja. Eso es una V invertida "
        "hecha con dos funciones que solas no saben bajar.",
        "por ahora la curva no baja después del pico. La segunda altura tiene "
        "que ser NEGATIVA y más grande (en valor absoluto) que la primera.")

### `TODO 3` — Ahora una meseta

Una **meseta**: sube, se queda plana un rato, y después baja.

Antes de escribir nada: **¿cuántas rampas necesitás?** Contá los cambios de
pendiente que ves en una meseta y escribí el número acá abajo. Después probalo.

In [ ]:
# mi respuesta: hacen falta ..... rampas   (escribila antes de tocar el código)

codos = [2.0, 4.0, 7.0]
alturas = [1.0, 0.0, 0.0]        # TODO: tocá estas tres alturas


def meseta(x):
    total = np.zeros_like(x)
    for codo, altura in zip(codos, alturas):
        total = total + rampa(x, codo, altura)
    return total


y = meseta(x)
g.plot_funcion(meseta, x=x, xlabel="x", ylabel="suma de las tres rampas",
               titulo="TODO 3: ¿sube, se queda plana, y baja?")

pendientes = np.diff(y) / np.diff(x)
subida = pendientes[(x[:-1] > codos[0]) & (x[:-1] < codos[1])].mean()
plano = pendientes[(x[:-1] > codos[1]) & (x[:-1] < codos[2])].mean()
bajada = pendientes[x[:-1] > codos[2]].mean()

print(f"pendiente entre {codos[0]} y {codos[1]}: {subida:+.2f}")
print(f"pendiente entre {codos[1]} y {codos[2]}: {plano:+.2f}")
print(f"pendiente después de {codos[2]}:        {bajada:+.2f}")

revisar(subida > 0.1 and abs(plano) < 0.05 and bajada < -0.1,
        "meseta. Tres rampas: una que la levanta, una que la frena, una que la baja.",
        "para que el tramo del medio quede PLANO, la segunda altura tiene que "
        "cancelar exactamente a la primera (probá el negativo de la primera). "
        "Y la tercera es la que hace bajar.")

---

## 4. Sumar muchas rampas: la red neuronal, entera, a mano

Y acá está el truco. ¿Qué pasa si sumamos muchas rampas, cada una con su codo
en un lugar distinto? Cada una se hace cargo de **un tramo** de la curva y no
se entera de lo que pasa en los otros.

La función de abajo tiene seis líneas. **Eso es una red neuronal con una capa
oculta.** No hay nada más. No es una metáfora ni una simplificación: es
literalmente la misma cuenta que hace PyTorch.

In [ ]:
def red_a_mano(x, codos, alturas, base=0.0):
    """Una red neuronal con una capa oculta. Sí, esto es todo."""
    total = np.full_like(np.asarray(x, dtype=float), base)
    for codo, altura in zip(codos, alturas):
        total = total + rampa(x, codo, altura)
    return total


# Cuatro rampas puestas a dedo, mirando dónde la serie cambia de dirección.
codos_4 = [1996, 2002, 2012, 2016]
alturas_4 = [-0.0053, 0.0039, 0.0290, -0.0286]
base_4 = 0.0764

prediccion = red_a_mano(anios, codos_4, alturas_4, base_4)
mse_4 = float(np.mean((inflacion - prediccion) ** 2))

print(f"desajuste con 4 rampas ........ {mse_4:.5f}")
print(f"desajuste de la MEJOR recta ... {mse_recta:.5f}")
print(f"o sea, {mse_recta / mse_4:.1f} veces mejor, con cuatro cosas tontas sumadas.")

g.plot_ajuste(anios, inflacion, modelo=lambda t: red_a_mano(t, codos_4, alturas_4, base_4),
              loss=mse_4, xlabel="año", ylabel="% de palabras",
              etiqueta_modelo="4 rampas sumadas",
              titulo="Cuatro rampas contra la curva imposible")

Y para que se vea de dónde sale la suma, dibujemos **cada rampa por separado**
y después el resultado. Ninguna entiende nada; juntas dibujan la figura.

In [ ]:
fig, ejes = g.figura(11, 4.2, filas=1, columnas=2,
                     titulo="Las piezas sueltas, y la suma")

colores = [g.PARAMETRO, g.ACENTO, g.MODELO, g.VAL]
for i, (codo, altura) in enumerate(zip(codos_4, alturas_4)):
    ejes[0].plot(anios, base_4 + rampa(anios, codo, altura), color=colores[i],
                 linewidth=2, linestyle=(0, (5, 4)),
                 label=f"codo en {codo}")
ejes[0].set_title("cada rampa sola, sumada a la base")
ejes[0].set_xlabel("año"); ejes[0].set_ylabel("% de palabras")
ejes[0].legend(fontsize=8)

g.plot_ajuste(anios, inflacion,
              modelo=lambda t: red_a_mano(t, codos_4, alturas_4, base_4),
              ax=ejes[1], mostrar_errores=False, etiqueta_modelo="la suma",
              titulo="las cuatro sumadas", xlabel="año", ylabel="% de palabras")
print("Izquierda: cuatro rectas quebradas que solas no dicen nada.")
print("Derecha: la misma suma, y ya se parece.")

### `TODO 4` — Bajá el desajuste

**Consigna.** Arrancás con las cuatro rampas de recién, que dan **0.00056**.
Agregá y movés rampas hasta **bajar de ahí**. Anotá cuántas necesitaste y hasta
dónde llegaste.

*Pista 1: mirá los palitos rojos del gráfico. Donde el palito es más largo,
falta un codo. Los dos tramos peores son alrededor de **2006-2007** y de
**2014**.*
*Pista 2: si el tramo tiene que bajar, la altura va negativa. Si tiene que
subir más fuerte, va positiva y más grande.*
*Pista 3: `base_mia` levanta o baja todo junto. Después de agregar una rampa,
volvé a acomodarla.*

**Para calibrar:** la mejor recta posible da **0.00150**. Con ocho rampas
*óptimamente* puestas se llega a **0.00014**, pero eso no se consigue a mano
(y ése es justamente el problema que resuelve la clase 3). Bajar de **0.00040**
a mano ya es muy bueno.

In [ ]:
# ← toqueteá esto. Agregá elementos a las dos listas (tienen que tener el mismo largo).
codos_mios = [1996, 2002, 2012, 2016]
alturas_mios = [-0.0053, 0.0039, 0.0290, -0.0286]
base_mia = 0.0764

pred = red_a_mano(anios, codos_mios, alturas_mios, base_mia)
mse_mio = float(np.mean((inflacion - pred) ** 2))

print(f"{len(codos_mios)} rampas  ->  desajuste {mse_mio:.5f}   "
      f"(objetivo: bajar de {mse_4:.5f}; abajo de 0.00040 es excelente)")
print(f"perillas usadas: {2 * len(codos_mios) + 1}   "
      f"(un codo y una altura por rampa, más la base)")

g.plot_ajuste(anios, inflacion,
              modelo=lambda t: red_a_mano(t, codos_mios, alturas_mios, base_mia),
              loss=mse_mio, xlabel="año", ylabel="% de palabras",
              etiqueta_modelo=f"{len(codos_mios)} rampas",
              titulo="TODO 4: tu red")

revisar(len(codos_mios) == len(alturas_mios),
        "las dos listas tienen el mismo largo.",
        "`codos` y `alturas` tienen que tener el mismo largo: cada rampa "
        "necesita su codo Y su altura.")
revisar(mse_mio < mse_4,
        f"le ganaste a la configuración de arriba: {mse_mio:.5f} contra "
        f"{mse_4:.5f}, con {len(codos_mios)} rampas y "
        f"{2 * len(codos_mios) + 1} perillas movidas a mano.",
        f"vas {mse_mio:.5f} y arriba teníamos {mse_4:.5f}. Mirá el gráfico y "
        "buscá el tramo con los palitos rojos más largos: ahí falta un codo.")

if mse_mio < 0.00040:
    print("★ Y encima bajaste de 0.00040 a mano. Eso es mucho más difícil de lo")
    print("  que parece: acordate de cuánto te costó cuando en la clase 3 veas")
    print("  a la máquina hacerlo sola en dos segundos.")

---

## 5. Lo mismo, escrito con matrices *(opcional, pero conviene)*

Si abrís cualquier paper o cualquier librería, la red no está escrita con un
`for`: está escrita con un producto de matrices. **No es otra cosa.** Es la
misma cuenta, escrita corta.

$$\text{salida} = W_2 \cdot \text{relu}(W_1 x + b_1) + b_2$$

- $W_1$ y $b_1$ son las perillas de las rampas: cuánto se estira la entrada y
  dónde queda el codo.
- $\text{relu}$ es el doblez.
- $W_2$ son las alturas, y $b_2$ es la base.

Al lado, la figura: la de arriba ya la viste; es la misma. Y el `assert` de
abajo comprueba que las dos versiones dan **el mismo número, bit por bit**.

In [ ]:
# Las mismas perillas de recién, ahora como vectores.
W1 = np.ones(len(codos_4))                 # cuánto estira la entrada cada rampa
b1 = -np.array(codos_4, dtype=float)       # el codo, escrito como un desplazamiento
W2 = np.array(alturas_4)                   # las alturas
b2 = base_4                                # la base


def red_matricial(x):
    x = np.asarray(x, dtype=float).reshape(-1, 1)     # (n, 1)
    oculta = np.maximum(0, x * W1 + b1)               # (n, 4)  <- el doblez
    return oculta @ W2 + b2                           # (n,)


con_for = red_a_mano(anios, codos_4, alturas_4, base_4)
con_matrices = red_matricial(anios)

diferencia = np.abs(con_for - con_matrices).max()
print(f"mayor diferencia entre las dos versiones: {diferencia:.2e}")
print("(o sea: un 7 en el decimoctavo decimal. La computadora suma los números")
print(" en otro orden y redondea distinto en el último lugar que sabe")
print(" representar. Para cualquier uso, eso es cero.)")

assert diferencia < 1e-12, (
    "Las dos versiones tendrían que dar lo mismo. "
    "Revisá que W1, b1, W2 y b2 salgan de las MISMAS listas codos_4 y alturas_4."
)
print("\nSon la misma cuenta. La notación matricial no es un concepto nuevo:")
print("es taquigrafía. En la clase 4, PyTorch va a hacer exactamente esto.")

print(f"\nEsta red es una [1, {len(codos_4)}, 1]: "
      f"{len(W1)} + {len(b1)} + {len(W2)} + 1 = {2 * len(codos_4) + len(W2) + 1} parámetros.")

### `TODO 5` *(opcional)* — Cambiale el doblez

La rampa no es el único doblez posible. La **sigmoide** de la clase 1 también
dobla, sólo que suave. Cambiá `np.maximum(0, ...)` por la sigmoide y mirá la
figura que sale.

**Antes de correr:** ¿la curva va a quedar más angulosa o más redondeada? ¿Por qué?

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def ajustar_alturas(piezas):
    """Busca de una las mejores alturas para estas piezas.

    Es la misma fórmula cerrada de `np.polyfit`, con más columnas. Y sí, es
    hacer trampa: en la clase 3 vamos a aprender a encontrarlas a pasitos,
    que es como se hace cuando no hay fórmula cerrada (o sea, casi siempre).
    """
    A = np.column_stack([np.ones(piezas.shape[0]), piezas])
    coeficientes, *_ = np.linalg.lstsq(A, inflacion, rcond=None)
    return coeficientes


# El mismo esqueleto —los mismos codos— con dos dobleces distintos.
z = anios.reshape(-1, 1) * W1 + b1        # lo que le entra al doblez
piezas_rampa = np.maximum(0, z)           # el doblez brusco
piezas_sigmo = sigmoide(0.25 * z)         # el doblez suave

coef_rampa = ajustar_alturas(piezas_rampa)
coef_sigmo = ajustar_alturas(piezas_sigmo)


def dibujar_con(piezas_de, coeficientes):
    def f(x):
        x = np.asarray(x, dtype=float).reshape(-1, 1)
        piezas = piezas_de(x * W1 + b1)
        return np.column_stack([np.ones(len(piezas)), piezas]) @ coeficientes
    return f


red_rampas = dibujar_con(lambda z: np.maximum(0, z), coef_rampa)
red_sigmoides = dibujar_con(lambda z: sigmoide(0.25 * z), coef_sigmo)

fig, ejes = g.figura(11, 4.2, filas=1, columnas=2, titulo="El mismo esqueleto, otro doblez")
g.plot_ajuste(anios, inflacion, modelo=red_rampas, ax=ejes[0],
              mostrar_errores=False, etiqueta_modelo="con rampas (ReLU)",
              titulo="con rampas: codos", xlabel="año", ylabel="% de palabras")
g.plot_ajuste(anios, inflacion, modelo=red_sigmoides, ax=ejes[1],
              mostrar_errores=False, etiqueta_modelo="con sigmoides",
              titulo="con sigmoides: curvas", xlabel="año", ylabel="% de palabras")

print(f"desajuste con rampas ....... {np.mean((inflacion - red_rampas(anios))**2):.5f}")
print(f"desajuste con sigmoides .... {np.mean((inflacion - red_sigmoides(anios))**2):.5f}")
print()
print("Con rampas la curva tiene CODOS: es una poligonal.")
print("Con sigmoides es REDONDEADA, porque cada pieza dobla despacio.")
print("Las dos dibujan cualquier figura. La rampa se usa más porque es")
print("muchísimo más barata de calcular, y porque anda. Gran parte del deep")
print("learning es así, y está bien saberlo.")

---

## 6. Qué te llevás

**Las tres cosas de hoy:**

1. **Componer es encadenar**: la salida de una función entra a la siguiente. Y
   el orden importa, porque componer no es sumar.
2. **Apilar cosas lineales no sirve de nada.** Cinco cajas lineales seguidas se
   colapsan a dos números. Hace falta un **doblez** en el medio, y el doblez que
   se usa es `max(0, x)`: no hay nada que entender ahí adentro.
3. **Una red neuronal es una suma de rampas** con las alturas y los codos
   regulables. Con suficientes rampas dibujás cualquier figura. Pero *poder
   dibujar cualquier figura* no es lo mismo que *predecir bien*: esa distinción
   es la clase 4, y es la confusión más cara de todas.

**El vocabulario nuevo** (también está en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *composition* | composición — la salida de una es la entrada de la otra |
| *non-linearity* | no linealidad — cualquier cosa que doble la línea |
| *ReLU* | la rampa: `max(0, x)`. Si es negativo, cero |
| *activation function* | función de activación — el doblez que va entre capa y capa |
| *weight* | peso — la perilla que multiplica |
| *bias* | sesgo — la perilla que desplaza. **Nada que ver con sesgo social** |
| *neuron* | neurona — una rampa. No es una célula ni piensa |
| *hidden layer* | capa oculta — el grupo de rampas del medio |
| *universal approximation* | aproximación universal — con suficientes rampas, cualquier figura |

**Para el lunes, en tu propia investigación.** Cuando leas *"usamos una red
neuronal"*, traducí mentalmente: *"usamos una función muy flexible con
muchísimas perillas"*. Y hacé las dos preguntas de revisor:
**¿cuántas perillas?** y **¿cuántos datos?** Un modelo con más perillas que
datos puede dibujar cualquier cosa, incluido el ruido.

**Lo que queda prometido.** Vos pusiste estas perillas a mano y fue un
suplicio: cuatro rampas son nueve números y ya cuesta. Un modelo de verdad
tiene millones. **¿Cómo los encuentra solo?** Eso es la clase 3: un número que
mide qué tan mal está, y una brújula que dice para dónde ir.

---

### Para seguir

Las referencias de la clase están en `referencias.md`. La primera de la lista es
el capítulo 4 de Nielsen: es exactamente esta clase, con visualizaciones, y está
gratis.